# Maritime fuels → CH3OH substitution scenarios (CONCAWE proxy)

This notebook reads `maritime_methanol_config.yaml`, reconstructs a proxy of marine fuel production from CONCAWE, applies demand scenarios and a linear methanol penetration, then writes scenario CSVs.

In [3]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import os


In [4]:
# =============================================================================
# LOAD YAML CONFIG
# =============================================================================

CONFIG_PATH = Path(os.environ.get("CONFIG_PATH", "maritime_methanol_config.yaml"))


with CONFIG_PATH.open("r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

YEAR_START = int(CFG["years"]["start"])
YEAR_END = int(CFG["years"]["end"])
REFERENCE_YEAR = int(CFG["years"]["reference_year"])
YEARS = np.arange(YEAR_START, YEAR_END + 1)

REF_CFG = CFG["refinery_inputs"]
MAR_CFG = CFG["maritime_proxy"]
SUB_CFG = CFG["substitution"]
EN_CFG = CFG["energy"]
MEOH_CFG = CFG["methanol_parameters"]
DEMAND_CFG = CFG["demand_scenarios"]
OUT_CFG = CFG["output"]
CHECKS = CFG["checks"]

OUT_DIR = Path(OUT_CFG["base_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Key parameters
PROXY_UNIT = str(MAR_CFG["proxy_unit"])
PROXY_COEFF = float(MAR_CFG["proxy_output_coeff_tFuel_per_tFeed"])

LHV_MARINE = float(EN_CFG["lhv_marine_fuel_mj_per_kg"])
LHV_MEOH = float(EN_CFG["lhv_methanol_mj_per_kg"])
USE_EN_EQ = bool(EN_CFG.get("use_energy_equivalence", True))

H2_INTENSITY_MEOH = float(MEOH_CFG["h2_intensity_tH2_per_tMeOH"])

TARGET_SHARE_2050 = float(SUB_CFG["methanol_penetration"]["target_share_2050"])
SUB_MODE = str(SUB_CFG.get("mode", "linear_share_with_concawe_cap"))


In [6]:
# =============================================================================
# FUNCTIONS
# =============================================================================

def read_refinery(path: Path) -> pd.DataFrame:
    """Read CONCAWE refinery file using YAML schema, return standardized columns."""
    df = pd.read_csv(path)

    schema = REF_CFG["schema"]
    df = df.rename(
        columns={
            schema["country_column"]: "country",
            schema["year_column"]: "year",
            schema["unit_column"]: "unit",
            schema["feed_column"]: "unit_feed",
        }
    )

    required = {"country", "year", "unit", "unit_feed"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {missing}")

    df = df.copy()
    df["country"] = df["country"].astype(str).str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    df["unit_feed"] = pd.to_numeric(df["unit_feed"], errors="coerce").fillna(0.0)
    # fix typo if any in column name normalization
    if "unit_feed" in df.columns:
        df = df.rename(columns={"unit_feed": "unit_feed"})

    df["unit_feed"] = pd.to_numeric(df["unit_feed"], errors="coerce").fillna(0.0)

    return df[df["year"].isin(YEARS)]


def compute_marine_fossil_proxy(df: pd.DataFrame) -> pd.DataFrame:
    """
    Proxy marine fuel fossil production (t/yr) from Diesel Hydrotreater feed:
        marine_fossil_proxy = coeff * sum(unit_feed where unit == PROXY_UNIT)
    """
    g = df.groupby(["country", "year", "unit"], as_index=False)["unit_feed"].sum()
    tmp = g[g["unit"] == PROXY_UNIT].copy()

    if tmp.empty:
        raise ValueError(
            f"Proxy unit '{PROXY_UNIT}' not found in refinery file (after renaming). "
            "Check maritime_proxy.proxy_unit in YAML."
        )

    out = (
        tmp.groupby(["country", "year"], as_index=False)["unit_feed"]
        .sum()
        .rename(columns={"unit_feed": "marine_fossil_proxy_t_per_yr"})
    )
    out["marine_fossil_proxy_t_per_yr"] = PROXY_COEFF * out["marine_fossil_proxy_t_per_yr"].astype(float)
    return out


def methanol_share(year: int) -> float:
    """Linear methanol penetration from reference_year to end_year."""
    if year <= REFERENCE_YEAR:
        return 0.0
    span = max(1, YEAR_END - REFERENCE_YEAR)
    s = TARGET_SHARE_2050 * (year - REFERENCE_YEAR) / span
    return float(min(TARGET_SHARE_2050, max(0.0, s)))


def build_maritime_methanol_scenario(marine_proxy_df: pd.DataFrame, growth_rate: float) -> pd.DataFrame:
    """
    Build one scenario per country/year.

    Definitions (all in t/yr unless stated):
      marine_reference = marine_fossil_proxy(reference_year)
      marine_total(t) = marine_reference * (1+g)^(t-reference_year)

      target fossil share via methanol_share(t):
        marine_fossil_target = marine_total * (1 - share)

      concawe cap + balancing:
        marine_fossil_used = min(marine_fossil_proxy, marine_fossil_target)
        marine_methanol_eq = marine_total - marine_fossil_used   (equivalent marine fuel, t)

      energy conversion:
        methanol_t = marine_methanol_eq * (LHV_marine/LHV_meoh)
        h2_for_methanol = methanol_t * H2_INTENSITY_MEOH
    """
    rows = []

    for country, g in marine_proxy_df.groupby("country"):
        g = g.set_index("year").sort_index()

        if REFERENCE_YEAR not in g.index:
            # no reference, cannot anchor scenario
            continue

        marine_ref = float(g.loc[REFERENCE_YEAR, "marine_fossil_proxy_t_per_yr"])
        if marine_ref < 0:
            continue

        for year in YEARS:
            if year not in g.index:
                continue

            marine_proxy = float(g.loc[year, "marine_fossil_proxy_t_per_yr"])
            marine_total = marine_ref * ((1.0 + float(growth_rate)) ** (year - REFERENCE_YEAR))

            share = methanol_share(int(year))
            fossil_target = marine_total * (1.0 - share)

            if SUB_MODE != "linear_share_with_concawe_cap":
                raise ValueError(f"Unsupported substitution.mode: {SUB_MODE}")

            fossil_used = min(marine_proxy, fossil_target)
            methanol_eq = marine_total - fossil_used

            if USE_EN_EQ:
                methanol_t = methanol_eq * (LHV_MARINE / LHV_MEOH)
            else:
                # fallback: 1-to-1 mass (not recommended)
                methanol_t = methanol_eq

            h2_for_meoh = methanol_t * H2_INTENSITY_MEOH

            rows.append(
                {
                    "country": country,
                    "year": int(year),
                    "marine_fossil_proxy_t_per_yr": marine_proxy,
                    "marine_total_t_per_yr": marine_total,
                    "marine_fossil_used_t_per_yr": fossil_used,
                    "marine_methanol_eq_t_per_yr": methanol_eq,
                    "methanol_t_per_yr": methanol_t,
                    "h2_for_methanol_t_per_yr": h2_for_meoh,
                    "methanol_share": share,
                }
            )

    return pd.DataFrame(rows)


In [7]:
# =============================================================================
# CHECKS
# =============================================================================

def run_checks(df: pd.DataFrame):
    if df.empty:
        raise ValueError("Empty output dataframe (no countries with full data / reference year).")

    if CHECKS.get("non_negative_flows", True):
        cols = [
            "marine_fossil_proxy_t_per_yr",
            "marine_total_t_per_yr",
            "marine_fossil_used_t_per_yr",
            "marine_methanol_eq_t_per_yr",
            "methanol_t_per_yr",
            "h2_for_methanol_t_per_yr",
            "methanol_share",
        ]
        bad = (df[cols] < -1e-9).any().any()
        if bad:
            raise ValueError("Negative flows detected.")

    if CHECKS.get("enforce_mass_balance", True):
        diff = df["marine_total_t_per_yr"] - (df["marine_fossil_used_t_per_yr"] + df["marine_methanol_eq_t_per_yr"])
        if (diff.abs() > 1e-6).any():
            raise ValueError("Mass balance violation detected (marine_total != fossil_used + methanol_eq).")

    if CHECKS.get("full_year_coverage", True):
        expected = len(YEARS)
        for country, g in df.groupby("country"):
            if len(g["year"].unique()) != expected:
                raise ValueError(f"Incomplete year coverage for {country}: {len(g['year'].unique())}/{expected}")


In [8]:
# =============================================================================
# MAIN
# =============================================================================

COLMAP = OUT_CFG.get("columns", {})

def rename_output_columns(df: pd.DataFrame, refinery_scenario: str, demand_scenario: str) -> pd.DataFrame:
    df = df.copy()
    df[COLMAP.get("refinery_scenario", "refinery_scenario")] = refinery_scenario
    df[COLMAP.get("demand_scenario", "demand_scenario")] = demand_scenario

    # internal -> output mapping
    mapping = {
        "marine_fossil_proxy_t_per_yr": COLMAP.get("marine_fossil_proxy", "marine_fossil_proxy_t_per_yr"),
        "marine_total_t_per_yr": COLMAP.get("marine_total", "marine_total_t_per_yr"),
        "marine_fossil_used_t_per_yr": COLMAP.get("marine_fossil_used", "marine_fossil_used_t_per_yr"),
        "marine_methanol_eq_t_per_yr": COLMAP.get("marine_methanol_eq", "marine_methanol_eq_t_per_yr"),
        "methanol_t_per_yr": COLMAP.get("methanol_mass", "methanol_t_per_yr"),
        "h2_for_methanol_t_per_yr": COLMAP.get("h2_methanol", "h2_for_methanol_t_per_yr"),
        "methanol_share": COLMAP.get("methanol_share", "methanol_share"),
    }
    df = df.rename(columns=mapping)

    # reorder (best-effort)
    ordered = [
        "country",
        "year",
        COLMAP.get("refinery_scenario", "refinery_scenario"),
        COLMAP.get("demand_scenario", "demand_scenario"),
        mapping["marine_fossil_proxy_t_per_yr"],
        mapping["marine_total_t_per_yr"],
        mapping["marine_fossil_used_t_per_yr"],
        mapping["marine_methanol_eq_t_per_yr"],
        mapping["methanol_t_per_yr"],
        mapping["h2_for_methanol_t_per_yr"],
        mapping["methanol_share"],
    ]
    cols = [c for c in ordered if c in df.columns] + [c for c in df.columns if c not in ordered]
    return df[cols]


for ref_name, ref_info in REF_CFG["scenarios"].items():
    print(f"\nProcessing refinery scenario: {ref_name}")

    df_ref = read_refinery(Path(ref_info["path"]))
    marine_proxy_df = compute_marine_fossil_proxy(df_ref)

    for demand_name, demand_info in DEMAND_CFG.items():
        growth = float(demand_info["annual_growth_rate"])
        print(f"  → Building maritime CH3OH scenario: {demand_name}_{ref_name}")

        df_out = build_maritime_methanol_scenario(marine_proxy_df, growth)

        run_checks(df_out)

        # output folders: base_dir / refinery_scenario / demand_scenario
        out_path = OUT_DIR / ref_name / demand_name
        out_path.mkdir(parents=True, exist_ok=True)

        df_out2 = rename_output_columns(df_out, refinery_scenario=ref_name, demand_scenario=demand_name)

        df_out2.to_csv(out_path / OUT_CFG["output_file_name"], index=False)

print("\nAll maritime CH3OH substitution scenarios successfully written.")



Processing refinery scenario: more-molecule
  → Building maritime CH3OH scenario: constant-demand_more-molecule
  → Building maritime CH3OH scenario: growth-3pct_more-molecule
  → Building maritime CH3OH scenario: decline-1pct_more-molecule

Processing refinery scenario: max-electron
  → Building maritime CH3OH scenario: constant-demand_max-electron
  → Building maritime CH3OH scenario: growth-3pct_max-electron
  → Building maritime CH3OH scenario: decline-1pct_max-electron

All maritime CH3OH substitution scenarios successfully written.
